 ## Fase 1: Limpieza Estructural (Data Cleaning)
Antes de construir el grafo, deben usar pandas y las técnicas que vimos en la Unidad I y II para purificar la tabla.

- Valores Nulos: Encuentren y eliminen cualquier registro que no tenga Origen o Destino.
- Formato Inconsistente: El código del aeropuerto siempre debe ser de 3 letras mayúsculas (Ej. "MEX"). Arreglen las filas que estén en minúsculas.
- Anomalías Lógicas (Self-Loops): Un avión no puede despegar de un aeropuerto y aterrizar en ese mismo aeropuerto.
¿Qué debemos hacer con esos registros?

- Errores de Captura: La distancia no puede ser un número negativo, corrijan esos errores.
Duplicados: Para modelar la red, solo necesitamos saber si existe la ruta. Eliminen las filas duplicadas (mismo origen y mismo destino).


In [ ]:
import pandas as pd

# Cargar el archivo CSV en un DataFrame
df = pd.read_csv('vuelos_10k.csv')
df = df.dropna(subset=['Origen','Destino'])
df["Origen"] = df["Origen"].str.upper()
df["Destino"] = df["Destino"].str.upper()
df = df.drop_duplicates(subset=['Origen', 'Destino'])
df = df[df['Distancia_km'] >= 0]
df = df[df["Origen"] != df["Destino"]]
df.head(10)


,Origen,Destino,Distancia_km
0,RYI,KXK,965.34
1,ULE,PVC,1098.58
2,LTV,MOV,2713.63
3,NFR,CLR,1138.99
4,TSF,CZE,1691.08
5,LLN,PSE,1739.30
7,VPJ,PNT,996.44
8,YML,NGU,1189.53
9,ULF,ARF,1207.08
10,EPQ,ITD,1690.62


Explicación fase 1: En la primera fase lo que hicimos fue importar las librerías necesarias para realizar la limpieza del dataset y el resto de la actividad. Una vez que el dataset esta limpio pudimos pasar a la siguiente fase.

### Fase 2: Extracción de la Red
Con su DataFrame perfectamente limpio (que actúa como una Lista de Aristas o Edge List):

Importen la librería networkx.
Utilicen la función nx.from_pandas_edgelist() para convertir su DataFrame en un Grafo Dirigido (nx.DiGraph).
Impriman en pantalla cuántos Nodos (Aeropuertos) y cuántas Aristas (Rutas) tiene su grafo final.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Crear el grafo directamente desde el DataFrame con sus pesos (Distancia_km)
G = nx.from_pandas_edgelist(df, source='Origen', target='Destino', edge_attr='Distancia_km', create_using=nx.DiGraph())

print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())


Número de nodos: 3253
Número de aristas: 9643


Explicación Fase 2: En esta segunda fase importamos las librerias necesarias para crear el grafo utilizando el dataframe. En este caso utilizamos la funcion nx.from_pandas_edgelist para crearlo y después mostramos la cantidad de nodos y de aristas creados.

## Fase 3: El Experimento de la Memoria RAM
¡Llegó la hora de comprobar la teoría de la clase!

Construyan la Matriz de Adyacencia densa a partir de su grafo usando nx.adjacency_matrix(G).todense().
Utilicen código Python (ej. el atributo .nbytes) para calcular cuántos Megabytes (MB) de Memoria RAM está consumiendo su Matriz de Adyacencia llena de ceros.
Calculen cuántos Megabytes está consumiendo su DataFrame limpio (la Lista de Aristas).
Veredicto: Añadan una celda Markdown al final de su notebook respondiendo: ¿Cuántas veces más pesada es la Matriz en comparación con la Lista de Aristas y por qué sucede esto?

In [ ]:
matriz = nx.adjacency_matrix(G).todense()
print("Nodos en orden:", list(G.nodes())[:10], "... (total:", len(G), ")")
print("\nMatriz (Dimensiones):", matriz.shape)
print("DataFrame bytes: ", df.memory_usage(deep=True).sum())
print("Bytes de matriz: ", matriz.nbytes)
diff = matriz.nbytes / df.memory_usage(deep=True).sum()
print("Diferencia: ", diff)
print(matriz)

Nodos en orden: ['RYI', 'KXK', 'ULE', 'PVC', 'LTV', 'MOV', 'NFR', 'CLR', 'TSF', 'CZE'] ... (total: 3253 )

Matriz (Dimensiones): (3253, 3253)
DataFrame bytes:  1157160
Bytes de matriz:  84656072
Diferencia:  73.15848456566076
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Explicación de la fase 3: En la tercera fase, creamos la matriz de adyacencia a partir del grafo creado en la fase previa. Aqui ademas de crear la matriz pudimos ver que la matriz original es bastante mas pesada a comparacion de la matriz limpia. Por ejemplo en este caso la matriz fue
73.158 veces más grande que cuando se calculó la matriz limpia.